In [1]:
import pandas as pd

In [43]:
# ~~~~ SCENARIO NAME ~~~~ #
scen_name = "test"

# ~~~~ DATABASE ~~~~ #
# Specify path and name of M3VTEF, M3GEFP, M3LDF databases to be created
M3GEFP_database_path = "../database/M3GEF_database."+scen_name+".db"

# ~~~~ INPUT TABLES ~~~~ #
# Directories with premade CSV files
# to be converted into database tables
inputs_path = "../inputs/"

# input file names
EF = "EF.txt"
Ecotype_Crop = "SpeciationCrop.txt"
Ecotype_Herb = "SpeciationHerb.txt"
Ecotype_Shrub = "SpeciationShrub.txt"
Ecotype_Tree = "SpeciationTree.txt"
grid_ecotype = "GridEcotype.txt"
grid_growth_form = "GridGrowthForm.txt"

# ~~~~ SETTINGS ~~~~ #
# Total number of classes to loop through
# Default: 20
# If non-default number used, updates must be made in all submodules' concat_*_tables function
TotalEFs = 18
TotalLDFs = [3,4,5,6]

# ~~~~ OUTPUTS ~~~~ #
# Output directory
outputs_path = "../output/"


In [60]:
"""
Module that contains functions related to createing the Grid EF database
"""

import sqlite3
import os
import errno
import warnings
import numpy as np
warnings.simplefilter(action="ignore", category=FutureWarning)

def make_dir(path):
    """

    Function to check if output path exists
    and create directories if needed
    :param path: directory path
    :return: new directory if one does not previously exists

    """
    try:
        os.makedirs(path)
    except OSError as exception:
        if exception.errno != errno.EEXIST:
            raise

def make_M3GEFP_tables(conn, csv_input_dir, Ecotype_Crop, Ecotype_Shrub, Ecotype_Herb,
                       Ecotype_Tree, grid_ecotype, grid_growth_form, EF):
    """

    Function to load CSV inputs to create primary tables of the M3GEFP database
    :param conn: SQLite database connection
    :param csv_input_dir: Directory of CSV inputs
    :param Ecotype_Crop: input csv file name
    :param Ecotype_Shrub: input csv file name
    :param Ecotype_Herb: input csv file name
    :param Ecotype_Tree: input csv file name
    :param grid_ecotype: input csv file name
    :param grid_growth_form: input csv file name
    :param Vegtype_EF_csv_input: Input CSVs generated by running the M3VTEF database
    :return: A SQLite database with primary tables

    """

    print("Creating M3GEFP DB tables from: %s\n" % csv_input_dir)
    load_table(csv_input_dir + EF, conn, "EF")
    load_table(csv_input_dir + Ecotype_Crop, conn, "SpeciationCrop")
    load_table(csv_input_dir + Ecotype_Shrub, conn, "SpeciationShrub")
    load_table(csv_input_dir + Ecotype_Herb, conn, "SpeciationHerb")
    load_table(csv_input_dir + Ecotype_Tree, conn, "SpeciationTree")
    load_table(csv_input_dir + grid_ecotype, conn, "GridEcotype")
    load_table(csv_input_dir + grid_growth_form, conn, "GridGrowthForm")
    

def load_table(inpath, conn, table_name):
    df = pd.read_csv(inpath)
    df.to_sql(table_name, conn, schema='sqlite', if_exists='replace')
    print(f"{table_name} Table Loaded from: %s" % inpath)

    
def build_interm_query(growthform,ef_s=1,ef_e=18,ldf_s=3,ldf_e=6):
    """
    Function to build the intermediate query
    """
    query_str = f"CREATE TABLE 'Intermediate{growthform}EcoEF' AS \
                  SELECT Speciation{growthform}.EcoTypeID, "
    for i in range(ef_s,ef_e+1,1):
        q = f"Sum([EF{i}]*[{growthform}Specfrac]) AS {growthform}EcoEF{i}, "
        query_str+=q
    for j in range(ldf_s,ldf_e+1,1):
        q2 = f"Sum([LDF{j}]*[{growthform}Specfrac]) AS {growthform}EcoLDF{j}, "
        query_str+=q2
    query_str = query_str.rstrip(', ')
    query_str+=f" FROM Speciation{growthform} \
                 INNER JOIN EF ON Speciation{growthform}.VegID = EF.VegID \
                 GROUP BY Speciation{growthform}.EcoTypeID;"
    return query_str


def Ecotype_Tree_EF(conn):
    c = conn.cursor()
    c.execute("DROP TABLE IntermediateTreeEcoEF")
    query_str = build_interm_query("Tree")
    print(query_str)
    c.execute(query_str)
    print("'IntermediateTreeEcoEF' Table Created")


def Ecotype_Shrub_EF(conn):
    c = conn.cursor()
    c.execute("DROP TABLE IntermediateShrubEcoEF")
    query_str = build_interm_query("Shrub")
    c.execute(query_str)
    print("'IntermediateShrubEcoEF' Table Created")
    
    
def Ecotype_Herb_EF(conn):
    c = conn.cursor()
    c.execute("DROP TABLE IntermediateHerbEcoEF")
    query_str = build_interm_query("Herb")
    c.execute(query_str)
    print("'IntermediateHerbEcoEF' Table Created")
    

def Ecotype_Crop_EF(conn):
    c = conn.cursor()
    c.execute("DROP TABLE IntermediateCropEcoEF")
    query_str = build_interm_query("Crop")
    c.execute(query_str)
    print("'IntermediateCropEcoEF' Table Created")

    
def build_final_query(ef_s=1,ef_e=18,ldf_s=3,ldf_e=6):
    query_str = "CREATE TABLE 'OutputGridEF' AS \
                SELECT GridGrowthForm.gridID, "
    for i in range(1,19,1):
        q = f"Sum([EcotypeFrac]*(([CropFrac]*[CropEcoEF{i}])\
        +([TreeFrac]*[TreeEcoEF{i}])\
        +([HerbFrac]*[HerbEcoEF{i}])\
        +([ShrubFrac]*[ShrubEcoEF{i}]))) AS EF{i}, "
        query_str += q

    for j in range(3,7,1):
        q2 = f"Sum([EcotypeFrac]*(([CropFrac]*[CropEcoLDF{j}])\
        +([TreeFrac]*[TreeEcoLDF{j}])\
        +([HerbFrac]*[HerbEcoLDF{j}])\
        +([ShrubFrac]*[ShrubEcoLDF{j}]))) AS LDF{j}, "
        query_str += q2

    query_str = query_str.rstrip(', ')
    query_str += " FROM ((((GridGrowthForm INNER JOIN GridEcotype ON GridGrowthForm.gridID = GridEcotype.gridID)  \
    INNER JOIN IntermediateHerbEcoEF ON GridEcotype.EcotypeID = IntermediateHerbEcoEF.EcoTypeID)                  \
    INNER JOIN IntermediateShrubEcoEF ON IntermediateHerbEcoEF.EcoTypeID = IntermediateShrubEcoEF.EcoTypeID)      \
    INNER JOIN IntermediateTreeEcoEF ON IntermediateShrubEcoEF.EcoTypeID = IntermediateTreeEcoEF.EcoTypeID)       \
    INNER JOIN IntermediateCropEcoEF ON IntermediateTreeEcoEF.EcoTypeID = IntermediateCropEcoEF.EcoTypeID         \
    GROUP BY GridGrowthForm.gridID;"

    return query_str


def grid_EF(conn):
    c = conn.cursor()
    query_str = build_final_query()
    c.execute(query_str)
    print("'OutputGridEF' Table Created")
    grid_EF_df = pd.read_sql_query("SELECT * FROM 'OutputGridEF';", conn)
    return grid_EF_df


def run_M3GEFP_DB(db_connection, output_csv_path):
    """
    Function to create M3GEFP database and output the Grid EF table as a CSV
    :param db_connection: Database connection
    :param output_csv_path: Path to CSV output
    :return: A SQLite database and copy of the grid EF table as a CSV file
    """
    # Queries need to be executed in specific order
    # due to dependency of tables generated from previous queries
    print("\n BEGINNING M3GEFP DB GENERATION")
    Ecotype_Tree_EF(db_connection)
    Ecotype_Shrub_EF(db_connection)
    Ecotype_Herb_EF(db_connection)
    Ecotype_Crop_EF(db_connection)
    grid_EF_output = grid_EF(db_connection)

    grid_EF_output.to_csv(output_csv_path, index=False)  # Save EF zone table to CSV
    print("grid EF Table CSV Generated: %s" % output_csv_path)
                                                                                       

#def m3efp_driver():
#    M3GEFP_connection = sqlite3.connect(M3GEFP_database_path)
#    M3GEFP_connection.text_factory = str
#    mgefp.run_M3GEFP_DB(M3GEFP_connection, grid_EF_output)

In [36]:
#intermediate query
growthform = 'Tree'
ef_s = 1
ef_e = 18
ldf_s = 3
ldf_e = 6

query_str = f"CREATE TABLE 'Intermediate{growthform}EcoEF' AS \
              SELECT Speciation{growthform}.EcoTypeID, "
for i in range(ef_s,ef_e+1,1):
    q = f"Sum([EF{i}]*[{growthform}Specfrac]) AS {growthform}EcoEF{i}, "
    query_str+=q
for j in range(ldf_s,ldf_e+1,1):
    q2 = f"Sum([LDF{j}]*[{growthform}Specfrac]) AS {growthform}EcoLDF{j}, "
    query_str+=q2
query_str = query_str.rstrip(', ')
query_str+=" FROM Speciation{growthform} \
             INNER JOIN EF ON Speciation{growthform}.VegID = EF.VegID \
             GROUP BY Speciation{grownthform}.EcoTypeID;"

In [39]:
#final query
query_str = "SELECT GridGrowthForm.gridID, "
for i in range(1,19,1):
    q = f"Sum([EcotypeFrac]*(([CropFrac]*[CropEcoEF{i}])+([TreeFrac]*[TreeEcoEF{i}])+([HerbFrac]*[HerbEcoEF{i}])+([ShrubFrac]*[ShrubEcoEF{i}]))) AS EF{i}, "
    query_str += q
    
for j in range(3,7,1):
    q2 = f"Sum([EcotypeFrac]*(([CropFrac]*[CropEcoLDF{j}])+([TreeFrac]*[TreeEcoLDF{j}])+([HerbFrac]*[HerbEcoLDF{j}])+([ShrubFrac]*[ShrubEcoLDF{j}]))) AS LDF{j}, "
    query_str += q2

query_str = query_str.rstrip(', ')
query_str += " FROM ((((GridGrowthForm INNER JOIN GridEcotype ON GridGrowthForm.gridID = GridEcotype.gridID)  \
INNER JOIN IntermediateHerbEcoEF ON GridEcotype.EcotypeID = IntermediateHerbEcoEF.EcoTypeID)                  \
INNER JOIN IntermediateShrubEcoEF ON IntermediateHerbEcoEF.EcoTypeID = IntermediateShrubEcoEF.EcoTypeID)      \
INNER JOIN IntermediateTreeEcoEF ON IntermediateShrubEcoEF.EcoTypeID = IntermediateTreeEcoEF.EcoTypeID)       \
INNER JOIN IntermediateCropEcoEF ON IntermediateTreeEcoEF.EcoTypeID = IntermediateCropEcoEF.EcoTypeID         \
GROUP BY GridGrowthForm.gridID;"
    

In [44]:
M3GEFP_connection = sqlite3.connect(M3GEFP_database_path)

In [45]:
M3GEFP_connection.text_factory = str

In [46]:
make_M3GEFP_tables(M3GEFP_connection, inputs_path, Ecotype_Crop, Ecotype_Shrub,
                    Ecotype_Herb, Ecotype_Tree, grid_ecotype, grid_growth_form,EF)

Creating M3GEFP DB tables from: ../inputs/



/usr/people/yshi/.local/lib/python3.6/site-packages/pandas/core/generic.py:2615: UserWarning: The spaces in these column names will not be changed. In pandas versions < 0.14, spaces were converted to underscores.
  method=method,


EF Table Loaded from: ../inputs/EF.txt
SpeciationCrop Table Loaded from: ../inputs/SpeciationCrop.txt
SpeciationShrub Table Loaded from: ../inputs/SpeciationShrub.txt
SpeciationHerb Table Loaded from: ../inputs/SpeciationHerb.txt
SpeciationTree Table Loaded from: ../inputs/SpeciationTree.txt
GridEcotype Table Loaded from: ../inputs/GridEcotype.txt
GridGrowthForm Table Loaded from: ../inputs/GridGrowthForm.txt


In [61]:
grid_EF_output = './test_output.csv'
run_M3GEFP_DB(M3GEFP_connection, grid_EF_output)


 BEGINNING M3GEFP DB GENERATION
CREATE TABLE 'IntermediateTreeEcoEF' AS                   SELECT SpeciationTree.EcoTypeID, Sum([EF1]*[TreeSpecfrac]) AS TreeEcoEF1, Sum([EF2]*[TreeSpecfrac]) AS TreeEcoEF2, Sum([EF3]*[TreeSpecfrac]) AS TreeEcoEF3, Sum([EF4]*[TreeSpecfrac]) AS TreeEcoEF4, Sum([EF5]*[TreeSpecfrac]) AS TreeEcoEF5, Sum([EF6]*[TreeSpecfrac]) AS TreeEcoEF6, Sum([EF7]*[TreeSpecfrac]) AS TreeEcoEF7, Sum([EF8]*[TreeSpecfrac]) AS TreeEcoEF8, Sum([EF9]*[TreeSpecfrac]) AS TreeEcoEF9, Sum([EF10]*[TreeSpecfrac]) AS TreeEcoEF10, Sum([EF11]*[TreeSpecfrac]) AS TreeEcoEF11, Sum([EF12]*[TreeSpecfrac]) AS TreeEcoEF12, Sum([EF13]*[TreeSpecfrac]) AS TreeEcoEF13, Sum([EF14]*[TreeSpecfrac]) AS TreeEcoEF14, Sum([EF15]*[TreeSpecfrac]) AS TreeEcoEF15, Sum([EF16]*[TreeSpecfrac]) AS TreeEcoEF16, Sum([EF17]*[TreeSpecfrac]) AS TreeEcoEF17, Sum([EF18]*[TreeSpecfrac]) AS TreeEcoEF18, Sum([LDF3]*[TreeSpecfrac]) AS TreeEcoLDF3, Sum([LDF4]*[TreeSpecfrac]) AS TreeEcoLDF4, Sum([LDF5]*[TreeSpecfrac]) AS Tree